In [1]:
#!/usr/bin/env Rscript
# -*- coding: utf-8 -*-
#
# STEP3.1 統合スクリプト（単一ファイル版）
# signlessCorr_MDS: 距離 1 - |cor| × 線形(cmdscale)/非線形(isoMDS)
# - OH/FP を一括処理
# - top3 / cumeig(>=0.8)
# - NbClust (ward.D2, Euclidean) × 6指標
# - 出力は STEP3.1/runs/STEP3.1_<YYYYMMDD-HHMM>_signlessCorr_MDS/ 配下の
#   inputs / outputs/{figures,tables,reports,models} / logs / runmeta に保存
#
# 使い方：
#   Rscript STEP3.1_signlessCorr_MDS_unified.R
#
# 依存：NbClust, ggplot2, ggrepel, scales, MASS, mclust, dplyr

suppressWarnings({
  if (!require(NbClust)) { install.packages("NbClust"); library(NbClust) }
  if (!require(ggplot2)) { install.packages("ggplot2"); library(ggplot2) }
  if (!require(ggrepel)) { install.packages("ggrepel"); library(ggrepel) }
  if (!require(scales))  { install.packages("scales");  library(scales) }
  if (!require(MASS))    { install.packages("MASS");    library(MASS) }     # isoMDS
  if (!require(mclust))  { install.packages("mclust");  library(mclust) }   # ARI
  if (!require(dplyr))   { install.packages("dplyr");   library(dplyr) }
})

# ---- 設定 ----
base_dir      <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
dataset_files <- c("preprocessed_features_OH.csv", "preprocessed_features_FP.csv")
index_list    <- c("silhouette","dunn","gap","ch","db","ptbiserial")
modes         <- c("linear","nonlinear")
dim_modes     <- c("top3","cumeig")
target_vars   <- c("PCEmax","Jsc","Voc","FF")

set.seed(42)

# === STEP3.1 ラン・ディレクトリ構成 ===
step_name <- "STEP3.1"
runid     <- format(Sys.time(), "%Y%m%d-%H%M")  # 例: 20251027-0130
root_out  <- file.path(base_dir, step_name, "runs", paste0(step_name, "_", runid, "_signlessCorr_MDS"))
dir.create(root_out, showWarnings = FALSE, recursive = TRUE)

# 標準サブフォルダ
dirs_needed <- file.path(
  root_out, c("inputs",
              file.path("outputs","reports"),
              file.path("outputs","figures"),
              file.path("outputs","tables"),
              file.path("outputs","models"),
              "logs","runmeta"))
invisible(lapply(dirs_needed, dir.create, showWarnings = FALSE, recursive = TRUE))

# 入力のコピー（存在するものだけ）
for (f in dataset_files) {
  p <- file.path(base_dir, f)
  if (file.exists(p)) file.copy(p, file.path(root_out, "inputs", f), overwrite = TRUE)
}

# 環境・runmeta・ログ
log_path <- file.path(root_out, "logs", paste0(step_name, "_", runid, ".log"))
summary_csv <- file.path(root_out, "outputs", "reports", paste0("summary_", step_name, "_", runid, ".csv"))

sink(log_path, split = TRUE)
on.exit({ try(sink(NULL), silent = TRUE) }, add = TRUE)

message(sprintf("[%s] %s started", Sys.time(), step_name))
message("base_dir: ", base_dir)
message("root_out: ", normalizePath(root_out))

writeLines(c(
  paste0("# ", step_name, " Run"),
  paste0("- Run ID: ", runid),
  paste0("- Created: ", format(Sys.time(), "%Y-%m-%d %H:%M:%S")),
  paste0("- Base: ", base_dir),
  "- Inputs: ", paste(dataset_files[file.exists(file.path(base_dir, dataset_files))], collapse = ", ")
), con = file.path(root_out, "runmeta", paste0("README_run_", runid, ".md")))

write.csv(data.frame(artifact = "init",
                     created_at = format(Sys.time(), "%Y-%m-%dT%H:%M:%S"),
                     runid = runid),
          summary_csv, row.names = FALSE)

summ_append <- function(tag) {
  df <- data.frame(artifact = tag,
                   created_at = format(Sys.time(), "%Y-%m-%dT%H:%M:%S"),
                   runid = runid,
                   stringsAsFactors = FALSE)
  suppressWarnings({
    old <- try(read.csv(summary_csv, stringsAsFactors = FALSE), silent = TRUE)
    if (inherits(old, "data.frame")) df <- rbind(old, df)
  })
  write.csv(df, summary_csv, row.names = FALSE)
}

# ---- ユーティリティ ----
read_numeric_explanatory <- function(path, target_vars){
  df <- read.delim(path, header = TRUE, sep = ",", row.names = 1,
                   as.is = TRUE, strip.white = FALSE)
  # 目的変数を除外
  expl_vars <- df[, !(colnames(df) %in% target_vars), drop = FALSE]
  # 文字列列の粗除去（英字を含むものを除外）
  is_char <- sapply(expl_vars, function(col) any(grepl("[A-Za-z]", col) & col != "NA", na.rm = TRUE))
  numData <- expl_vars[, !is_char, drop = FALSE]
  if (!ncol(numData)) stop("数値の説明変数列が見つかりません: ", path)
  return(numData)
}

prep_signless_linear_mds <- function(numData, k_max = 200){
  corM <- suppressWarnings(cor(numData, use = "pairwise.complete.obs"))
  corM[is.na(corM)] <- 0
  dist_mat <- 1 - abs(corM)
  dist_mat[dist_mat <= 0] <- 1e-6  # 0 距離の微小補正
  ddata <- as.dist(dist_mat)

  n_vars <- ncol(numData)
  cmd_k  <- min(max(1, n_vars - 1), k_max)
  mds_linear <- cmdscale(ddata, eig = TRUE, k = cmd_k)

  all_eigs <- mds_linear$eig
  pos_idx  <- which(all_eigs > 0)
  pos_eigs <- all_eigs[pos_idx]
  contrib  <- if (length(pos_eigs) > 0) pos_eigs / sum(pos_eigs) else numeric(0)
  list(ddata = ddata, mds_linear = mds_linear,
       all_eigs = all_eigs, contrib_pos = contrib)
}

decide_k_cumeig <- function(contrib_pos, thr = 0.8){
  if (length(contrib_pos) == 0) return(3)
  k <- which(cumsum(contrib_pos) >= thr)[1]
  if (is.na(k) || k < 2) k <- 3
  return(k)
}

run_nbclust <- function(X, index){
  tryCatch({
    NbClust(data = X, diss = NULL, distance = "euclidean",
            min.nc = 2, max.nc = min(25, nrow(X) - 1),
            method = "ward.D2", index = index)
  }, error = function(e) { warning(paste("NbClust failed:", index, "-", e$message)); NULL })
}

save_eig_plots <- function(suffix_tag, ts_tag, contrib_pos){
  if (length(contrib_pos) == 0) { warning("No positive eigenvalues; plots skipped."); return() }
  df <- data.frame(Dim = seq_along(contrib_pos),
                   Prop = as.numeric(contrib_pos))
  df$Cum <- cumsum(df$Prop)

  p1 <- ggplot(df, aes(Dim, Prop)) +
    geom_col(width = 0.85, alpha = 0.9) +
    geom_line(aes(y = Cum), linewidth = 1.1) +
    geom_point(aes(y = Cum), size = 2) +
    scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                       sec.axis = sec_axis(~ ., name = "Cumulative share")) +
    labs(title = "Scree plot (positive eigenvalues; signlessCorr distance)",
         subtitle = suffix_tag, x = "Dimension", y = "Variance share") +
    theme_minimal(base_size = 12) +
    theme(panel.grid.minor = element_blank(),
          axis.title = element_text(face = "bold"),
          plot.title = element_text(face = "bold"))
  ggsave(file.path(root_out, "outputs", "figures",
                   paste0("MDS_scree_", suffix_tag, "_", ts_tag, ".png")),
         p1, width = 7, height = 5, dpi = 300)

  topN <- min(50, nrow(df))
  p2 <- ggplot(df[1:topN, ], aes(Dim, Prop)) +
    geom_col(width = 0.85, alpha = 0.95) +
    labs(title = paste0("Top-", topN, " eigenvalue shares (signlessCorr)"),
         subtitle = suffix_tag, x = "Dimension", y = "Variance share") +
    scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
    theme_minimal(base_size = 12) +
    theme(panel.grid.minor = element_blank(),
          axis.title = element_text(face = "bold"),
          plot.title = element_text(face = "bold"))
  ggsave(file.path(root_out, "outputs", "figures",
                   paste0("MDS_bar_top", topN, "_", suffix_tag, "_", ts_tag, ".png")),
         p2, width = 7, height = 5, dpi = 300)
}

# ========= メイン：OH / FP を順に実行 =========
for (ifname in dataset_files) {
  suffix_tag <- if (grepl("OH", ifname, ignore.case = TRUE)) "OH" else "FP"
  out_dir <- file.path(root_out, "outputs", suffix_tag)
  dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
  ts_tag <- format(Sys.time(), "%Y%m%d_%H%M%S")

  message("\n==============================")
  message(sprintf("=== Processing: %s (%s) ===", ifname, suffix_tag))
  message("==============================")

  # 1) データ読み込み（説明変数のみ、英字列除外）
  in_path <- file.path(base_dir, ifname)
  numData <- read_numeric_explanatory(in_path, target_vars)

  # 2) 距離=1-|cor| → 線形MDS（eig付き） → 正固有値寄与
  prep <- prep_signless_linear_mds(numData)
  ddata       <- prep$ddata
  mds_linear  <- prep$mds_linear
  all_eigs    <- prep$all_eigs
  contrib_pos <- prep$contrib_pos

  # 固有値寄与 CSV（reports）
  df_eig_out <- data.frame(
    Dim           = seq_along(all_eigs),
    Eigenvalue    = all_eigs,
    Share_pos     = { z <- rep(NA_real_, length(all_eigs));
                      pos <- which(all_eigs > 0);
                      if (length(pos)) z[pos] <- all_eigs[pos] / sum(all_eigs[pos]); z },
    CumShare_pos  = { z <- rep(NA_real_, length(all_eigs));
                      pos <- which(all_eigs > 0);
                      if (length(pos)) { tmp <- all_eigs[pos]/sum(all_eigs[pos]); z[pos] <- cumsum(tmp) } ; z }
  )
  write.csv(df_eig_out,
            file.path(root_out, "outputs", "reports",
                      paste0("MDS_eigen_contributions_", suffix_tag, "_", ts_tag, ".csv")),
            row.names = FALSE)

  # スクリープロット等（figures）
  save_eig_plots(suffix_tag, ts_tag, contrib_pos)

  # 3) cumeig(>=0.8) 軸数
  k_cumeig <- decide_k_cumeig(contrib_pos, thr = 0.8)

  # 4) 座標計算（linear/nonlinear × top3/cumeig）
  coords_list <- list()
  # linear
  coords_list[["linear_top3"]]   <- cmdscale(ddata, eig = FALSE, k = 3)
  coords_list[["linear_cumeig"]] <- cmdscale(ddata, eig = FALSE, k = k_cumeig)
  # nonlinear
  coords_list[["nonlinear_top3"]]   <- MASS::isoMDS(ddata, k = 3)$points
  coords_list[["nonlinear_cumeig"]] <- MASS::isoMDS(ddata, k = k_cumeig)$points

  # 行名は変数名にし、座標CSVを保存（models）
  for (nm in names(coords_list)) {
    rownames(coords_list[[nm]]) <- colnames(numData)
    write.csv(coords_list[[nm]],
              file.path(root_out, "outputs", "models",
                        paste0("MDScoords_", nm, "_", suffix_tag, "_", ts_tag, ".csv")))
  }

  # 5) クラスタ割当のまとめテーブルと k サマリ
  var_ids <- colnames(numData)
  label_df <- data.frame(id = var_ids, stringsAsFactors = FALSE)
  ksum <- data.frame(mode = character(0), dim_mode = character(0),
                     index = character(0), k = integer(0), stringsAsFactors = FALSE)

  # 6) 各条件×指標で NbClust 実行・図保存・割当保存
  for (cond in names(coords_list)) {
    X <- coords_list[[cond]]
    mm <- strsplit(cond, "_")[[1]]; mode <- mm[1]; dim_mode <- mm[2]

    for (idx in index_list) {
      res <- run_nbclust(X, idx)
      col_name <- paste(cond, idx, sep = "_")
      if (is.null(res)) {
        label_df[[col_name]] <- NA_integer_
        ksum <- rbind(ksum, data.frame(mode = mode, dim_mode = dim_mode, index = idx, k = NA_integer_))
        next
      }

      # 割当
      grp <- as.factor(res$Best.partition)
      grp <- grp[var_ids]  # 行順合わせ
      label_df[[col_name]] <- as.integer(grp)

      # k
      k_val <- NA_integer_
      if (!is.null(res$Best.nc)) {
        if (is.list(res$Best.nc) && !is.null(res$Best.nc[["Number_clusters"]])) {
          k_val <- as.integer(res$Best.nc[["Number_clusters"]])
        } else if (is.matrix(res$Best.nc) && "Number_clusters" %in% rownames(res$Best.nc)) {
          k_val <- as.integer(res$Best.nc["Number_clusters", 1])
        } else if (is.numeric(res$Best.nc)) {
          k_val <- as.integer(res$Best.nc[1])
        }
      }
      ksum <- rbind(ksum, data.frame(mode = mode, dim_mode = dim_mode, index = idx, k = k_val))

      # 割当CSV（tables）
      write.csv(data.frame(Variable = names(grp), Cluster = grp),
                file.path(root_out, "outputs", "tables",
                          paste0("ClusterAssign_", cond, "_", idx, "_", suffix_tag, "_", ts_tag, ".csv")),
                row.names = FALSE)

      # All.index プロット（figures）
      if (!is.null(res$All.index)) {
        idx_vals <- as.numeric(res$All.index)
        k_labels <- names(res$All.index)
        k_seq <- if (is.null(k_labels) || any(k_labels == "")) seq_along(idx_vals) else suppressWarnings(as.numeric(k_labels))
        if (any(is.na(k_seq))) k_seq <- seq_along(idx_vals)
        df_idx <- data.frame(k = k_seq, value = idx_vals)
        p_idx <- ggplot(df_idx, aes(k, value)) +
          geom_line(linewidth = 1) + geom_point(size = 2) +
          labs(title = paste0("NbClust All.index — ", idx),
               subtitle = paste0(mode, " / ", dim_mode, "  |  ", suffix_tag),
               x = "k", y = "Index value") +
          theme_minimal(base_size = 12) +
          theme(panel.grid.minor = element_blank(),
                axis.title = element_text(face = "bold"),
                plot.title = element_text(face = "bold"))
        ggsave(file.path(root_out, "outputs", "figures",
               paste0("NbClust_AllIndex_", cond, "_", idx, "_", suffix_tag, "_", ts_tag, ".png")),
               p_idx, width = 7, height = 5, dpi = 300)
      }

      # 1-2軸散布図（figures）
      df_plot <- data.frame(MDS1 = X[,1], MDS2 = X[,2], Cluster = grp, ID = rownames(X))
      rng <- range(c(df_plot$MDS1, df_plot$MDS2), na.rm = TRUE)
      pad <- diff(rng) * 0.05; lims <- c(min(rng) - pad, max(rng) + pad)
      p_sc <- ggplot(df_plot, aes(MDS1, MDS2, color = Cluster)) +
        stat_ellipse(aes(group = Cluster), type = "norm", level = 0.95, linewidth = 0.6, alpha = 0.25) +
        geom_point(size = 2.2, alpha = 0.8) +
        coord_equal(xlim = lims, ylim = lims, expand = TRUE) +
        scale_color_hue(h = c(0,360), l = 55, c = 90) +
        labs(title = paste0("MDS (", mode, "/", dim_mode, ") — Ward.D2 × NbClust"),
             subtitle = paste0("Index: ", idx, "  |  k = ", k_val, "  |  ", suffix_tag),
             x = "Dim 1", y = "Dim 2", color = "Cluster") +
        theme_minimal(base_size = 12) +
        theme(panel.grid.major = element_line(linewidth = 0.3, linetype = "dashed"),
              panel.grid.minor = element_blank(),
              axis.title = element_text(face = "bold"),
              plot.title = element_text(face = "bold"))
      ggsave(file.path(root_out, "outputs", "figures",
             paste0("MDS12_scatter_", cond, "_", idx, "_", suffix_tag, "_", ts_tag, ".png")),
             p_sc, width = 7, height = 6, dpi = 300)
    }
  }

  # 7) ラベル一括・kサマリ（tables / reports）
  write.csv(label_df, file.path(root_out, "outputs", "tables",
         paste0("cluster_labels_", suffix_tag, "_", ts_tag, ".csv")), row.names = FALSE)
  write.csv(ksum,     file.path(root_out, "outputs", "reports",
         paste0("cluster_k_summary_", suffix_tag, "_", ts_tag, ".csv")), row.names = FALSE)

  # 8) ARI: top3 vs cumeig（reports）
  for (mode in c("linear","nonlinear")) {
    ari_rows <- lapply(index_list, function(ix){
      a <- label_df[[paste0(mode, "_top3_", ix)]]
      b <- label_df[[paste0(mode, "_cumeig_", ix)]]
      if (is.null(a) || is.null(b) || all(is.na(a)) || all(is.na(b))) {
        data.frame(Index = ix, ARI_top3_vs_cumeig = NA_real_)
      } else {
        data.frame(Index = ix,
                   ARI_top3_vs_cumeig = mclust::adjustedRandIndex(as.factor(a), as.factor(b)))
      }
    })
    df_ari <- dplyr::bind_rows(ari_rows)
    write.csv(df_ari, file.path(root_out, "outputs", "reports",
             paste0("ARI_top3_vs_cumeig_", mode, "_", suffix_tag, "_", ts_tag, ".csv")),
             row.names = FALSE)
  }

  message("\n✅ Done: ", suffix_tag, " → ", normalizePath(out_dir))
  summ_append(paste0("done_", suffix_tag))
}

message("\n🎉 All finished → ", normalizePath(root_out))
summ_append("finish_all")


Loading required package: NbClust

Loading required package: ggplot2

Loading required package: ggrepel

Loading required package: scales

Loading required package: MASS

Loading required package: mclust

Package 'mclust' version 6.1.1
Type 'citation("mclust")' for citing this R package in publications.

Loading required package: dplyr


Attaching package: 'dplyr'


The following object is masked from 'package:MASS':

    select


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


[2025-10-27 01:27:16.063569] STEP3.1 started

base_dir: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No

root_out: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.1/runs/STEP3.1_20251027-0127_signlessCorr_MDS



=== Processing: preprocessed_features_OH.csv (OH) ===




initial  value 76.771095 
final  value 76.771095 
converged
initial  value 15.266188 
iter   5 value 3.444006
iter  10 value 2.472963
iter  15 value 2.093113
iter  20 value 1.926440
iter  25 value 1.798742
iter  30 value 1.718089
iter  35 value 1.652814
iter  40 value 1.621325
iter  45 value 1.586389
iter  50 value 1.558392
final  value 1.558392 
stopped after 50 iterations


Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 9 rows containing missing values or values outside the scale range
(`geom_path()`)."
Too few points to calculate an ellipse
Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_path()`)."
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 9 rows conta

initial  value 46.616716 
iter   5 value 27.702921
iter  10 value 25.080133
iter  15 value 24.690498
final  value 24.569334 
converged
initial  value 11.422327 
iter   5 value 3.605791
iter  10 value 2.948323
iter  15 value 2.804869
iter  20 value 2.736545
final  value 2.705035 
converged


Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 3 rows containing missing values or values outside the scale range
(`geom_path()`)."
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 2 rows containing missing values or values outside the scale range
(`geom_path()`)."
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 2 rows containing missing values or values outside the scale range
(`geom_path()`)."
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 2 rows containing missing values or values outside the scale range
(`geom_path()`)."
Too few points to calculate an ellipse
Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_path()`)."
Too few points to calculate an ellipse
Warning message:
